## Spectrogram Generator

Reads `.wav` files and generates RGB Mel-spectrogram images `.png`. Each audio file is split into 11 segments of 10s with a 5s step and 5s overlap, each saved as a 512×512 image.

### Code structure

* **Libraries**
    - NumPy: array operations and zero-padding;
    - Matplotlib: plots and saves the images;
    - Librosa: audio loading and Mel-spectrogram computation;
    - OpenCV: image resizing;
    - gc: releases memory after each file.

* **`generate_spectrogram_segments(wav_path, output_folder)`**
    - Loads the audio at its native sample rate; corrupted files are reported and skipped;
    - Skips files already processed (last segment `_seg_11.png` exists), so interrupted runs can resume;
    - Slices the signal into segments (samples = seconds × sample rate); segments beyond the end of the audio are zero-padded;
    - Resizes the image to 512×512 using Lanczos interpolation (`INTER_LANCZOS4`);
    - Frees large variables and calls the garbage collector.

* **Main loop**
    - Iterates over the class folders, saves the images to a `Spectrograms_Output` subfolder and ignores files ending in `_pca.wav`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import cv2
import gc
import matplotlib
matplotlib.use('Agg')

## Spectrogram parameters

* `n_mels = 512`: number of Mel bands used for vertical resolution;
* `n_fft = 4096`: FFT window size; larger values increase frequency resolution;
* `hop_length = 512`: samples between consecutive frames determine temporal resolution;
* `ref = np.max`: dB scale normalized to the segment peak.

In [ ]:
def generate_spectrogram_segments(wav_path, output_folder):
    segment_duration = 10  
    overlap = 5            
    step = 5
    img_size = (512, 512)
    num_segments = 11

    try:
        y, sr = librosa.load(wav_path, sr=None)
    except Exception as e:
        print(f"ALERTA: O arquivo {os.path.basename(wav_path)} está corrompido e será ignorado.")
        return

    total_samples = len(y)
    base_name = os.path.basename(wav_path).replace('.wav', '')

    # Skip files already processed
    last_segment_path = os.path.join(output_folder, f"{base_name}_seg_{num_segments:02d}.png")
    if os.path.exists(last_segment_path):
        print(f"PULANDO: {base_name} já processado.")
        return 

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Split the audio into 11 overlapping segments
    for i in range(num_segments):
        start_sec = i * step
        end_sec = start_sec + segment_duration
        
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        
        if start_sample < total_samples: # Past the end of the audio -> zero segment
            segment = y[start_sample:end_sample]
        else:
            segment = np.zeros(int(segment_duration * sr))
        
        if len(segment) < int(segment_duration * sr):
            padding_needed = int(segment_duration * sr) - len(segment)
            segment = np.pad(segment, (0, padding_needed), mode='constant')

        # Mel-spectrogram
        S = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=512, n_fft=4096, hop_length=512)
        S_db = librosa.power_to_db(S, ref=np.max)

        # Plot
        dpi = 100
        fig_size = img_size[0] / dpi
        fig = plt.figure(figsize=(fig_size, fig_size), dpi=dpi)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.axis('off')

        librosa.display.specshow(S_db, sr=sr, x_axis=None, y_axis='mel', fmax=sr/2, ax=ax)

        output_filename = os.path.join(output_folder, f"{base_name}_seg_{i+1:02d}.png")
        plt.savefig(output_filename, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

        # Resize to 512x512
        img = cv2.imread(output_filename)
        if img is not None:
            img_resized = cv2.resize(img, img_size, interpolation=cv2.INTER_LANCZOS4)
            cv2.imwrite(output_filename, img_resized)

        plt.close(fig)
        fig.clf()
        del S, S_db, fig, ax

    del y
    gc.collect() 
    print(f"Finalizado: {base_name} -> 11 imagens geradas.")

In [ ]:
# Main processing loop

folders = [
    r'D:\Campus IDSM\no-rain',
    r'D:\Campus IDSM\light',
    r'D:\Campus IDSM\moderate',
    r'D:\Campus IDSM\heavy',
    r'D:\Campus IDSM\violent',
    ]
for folder_path in folders:
    if not os.path.exists(folder_path):
        continue
    output_dir = os.path.join(folder_path, "Spectrograms_Output")

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for filename in os.listdir(folder_path):
        if filename.endswith('.wav') and not filename.endswith('_pca.wav'):
            wav_full_path = os.path.join(folder_path, filename)
            print(f"Processando: {filename}...")
            generate_spectrogram_segments(wav_full_path, output_dir)

print("\nProcesso concluído com sucesso!")